# NSCH 2023–2025: SQL analysis (DuckDB)

Queries on the pooled 2023–2025 National Survey of Children's Health topical files. The same queries are saved as `.sql` files in `sql/`.

The database `data/processed/nsch.duckdb` is built by `src/build_db.py` (it is not committed, like the other data files).

In [1]:
import duckdb

con = duckdb.connect("../data/processed/nsch.duckdb", read_only=True)


def run(query):
    """Run a SQL query and return the result as a pandas table."""
    return con.sql(query).df()

## 1. Mother definition

Sampled children per survey year: all children, children whose respondent (Adult 1) is female, and children whose respondent is their biological or adoptive mother (`A1_SEX = 2 AND A1_RELATION = 1`).

Query file: `sql/01_mother_flow.sql`

In [2]:
mother_flow = run("""
SELECT
      SURVEY_YEAR AS survey_year,
      COUNT(*)                              AS all_children,
      COUNT(*) FILTER (WHERE A1_SEX = 2)        AS female_respondent,
      COUNT(*) FILTER (WHERE A1_SEX = 2 AND A1_RELATION = 1)        AS mothers
  FROM nsch
  GROUP BY survey_year
  ORDER BY survey_year;
""")
mother_flow

,survey_year,all_children,female_respondent,mothers
0,2023,55162,36334,33365
1,2024,51375,33656,30629
2,2025,45139,29021,26421


## 2. Sample sizes by child age band and employment

Unweighted number of mothers in each primary age band of the sampled child (0-5, 6-11, 12-17) by the mother's employment status (`A1_EMPLOYED_R`), pooled 2023–2025. Mothers with no employment answer are kept as `'Missing'`.

Query file: `sql/02_sample_sizes.sql`

In [3]:
sample_sizes = run("""
SELECT
    b.band_primary                 AS band,
    COALESCE(e.label, 'Missing')     AS employment,
    COUNT(*)                       AS n
FROM nsch AS c
JOIN age_bands AS b
    ON c.SC_AGE_YEARS = b.age
LEFT JOIN employment_labels AS e
    ON c.A1_EMPLOYED_R = e.code
WHERE c.A1_SEX = 2 AND c.A1_RELATION = 1
GROUP BY band, employment
ORDER BY band;
""")
sample_sizes

,band,employment,n
0,0-5,"Not employed, looking for work",1477
1,0-5,Employed part-time,4603
2,0-5,"Not employed, not looking for work",5247
3,0-5,Working without pay,431
4,0-5,Missing,234
5,0-5,Retired,62
6,0-5,Employed full-time,16552
7,12-17,Employed full-time,23274
8,12-17,Retired,623
9,12-17,Missing,385


## 3. Weighted employment shares

Weighted share (%) of mothers in each employment status within each age band, among mothers with an employment answer. Pooled weight = `FWC / 3`.

Query file: `sql/03_employment_shares.sql`

In [4]:
employment_shares = run("""
WITH totals AS (
      SELECT
          b.band_primary          AS band,
          e.label                 AS employment,
          SUM(c.FWC / 3)             AS weighted
      FROM nsch AS c
        JOIN age_bands AS b
            ON c.SC_AGE_YEARS = b.age
        INNER JOIN employment_labels AS e
            ON c.A1_EMPLOYED_R = e.code
        WHERE c.A1_SEX = 2 AND c.A1_RELATION = 1
        GROUP BY band, employment
        ORDER BY band
  )
  SELECT
      band,
      employment,
      ROUND(100 * weighted / SUM(weighted) OVER (PARTITION BY band), 1) AS share_pct
  FROM totals
  ORDER BY band, employment;
""")
employment_shares

,band,employment,share_pct
0,0-5,Employed full-time,51.6
1,0-5,Employed part-time,16.3
2,0-5,"Not employed, looking for work",8.7
3,0-5,"Not employed, not looking for work",21.5
4,0-5,Retired,0.3
5,0-5,Working without pay,1.6
6,12-17,Employed full-time,59.9
7,12-17,Employed part-time,15.8
8,12-17,"Not employed, looking for work",7.1
9,12-17,"Not employed, not looking for work",14.0


## 4. Missing-data codes

How often each missing-data code appears for each variable among mothers, from the `__MISS` columns (`m` no valid response, `l` logical skip, `n` not in universe, `d` suppressed for confidentiality).

**Finding:** `METRO_YN` (metro/non-metro) is suppressed (`d`) for 8,889 mothers (9.8%). The Census Bureau blanks it for every record in some states, so metro/non-metro comparisons exclude those states. See `docs/nsch-feasibility.md` Section 3.

Query file: `sql/04_missing_codes.sql`

In [5]:
missing_codes = run("""
 WITH mothers AS (
      SELECT COLUMNS( '.*_MISS')
      FROM nsch
      WHERE A1_SEX = 2 AND A1_RELATION = 1
  ),
  long AS (
      UNPIVOT mothers
      ON COLUMNS(*)
      INTO NAME variable VALUE code
  )
  SELECT
      REPLACE(variable, '__MISS', '')   AS variable,
      code,
      COUNT(*)                         AS n
  FROM long
  GROUP BY variable, code
  ORDER BY variable, code;
""")
missing_codes

,variable,code,n
0,A1_AGE,m,325
1,A1_EMPLOYED_R,m,821
2,A1_MARITAL,m,678
3,A1_MENTHEALTH,m,592
4,A1_PHYSHEALTH,m,623
5,A2_AGE,l,14262
6,A2_AGE,m,915
7,A2_EMPLOYED_R,l,14262
8,A2_EMPLOYED_R,m,1126
9,A2_GRADE,l,14262
